# Optional trained pair runtime — existing Kaggle session
Train the corrected ChangeVQA/TerraMind notebook first; attach its export as a private Kaggle input.
No training or artifact downloads happen here. Do not run a second GPU notebook just to keep a tunnel alive.
Fusion additionally requires terratorch from the fusion training environment.
Stop active requests before applying. Empty artifact paths deliberately load no pair expert.

In [ ]:
import sys, json, hashlib
from pathlib import Path
PAIR_RUNTIME_ROOT = Path('/kaggle/working/satquery_pair_runtime_v2')
PAIR_RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)
runtime_files = json.loads('{"satquery_ml/__init__.py": "", "satquery_ml/models/__init__.py": "", "satquery_model_service/__init__.py": "", "satquery_ml/models/notebook_experts.py": "\\"\\"\\"Generated by scripts/sync-expert-notebooks.py; architectures exactly match cloud training.\\"\\"\\"\\nimport math\\nimport torch\\nfrom torch.nn import functional as F\\nfrom torchvision.models import resnet18, ResNet18_Weights\\n\\nclass ChangeExpert(torch.nn.Module):\\n    def __init__(self, vocabulary_size, answer_classes, pretrained=True):\\n        super().__init__()\\n        encoder = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)\\n        self.stem = torch.nn.Sequential(encoder.conv1, encoder.bn1, encoder.relu, encoder.maxpool,\\n            encoder.layer1, encoder.layer2, encoder.layer3, encoder.layer4)\\n        self.fuse = torch.nn.Sequential(torch.nn.Conv2d(2048, 512, 1, bias=False),\\n            torch.nn.BatchNorm2d(512), torch.nn.GELU(), torch.nn.Conv2d(512, 256, 3, padding=1),\\n            torch.nn.GELU())\\n        self.embedding = torch.nn.Embedding(vocabulary_size, 256, padding_idx=0)\\n        self.question = torch.nn.GRU(256, 256, batch_first=True, bidirectional=True)\\n        self.answer_head = torch.nn.Sequential(torch.nn.Linear(768, 512), torch.nn.GELU(),\\n            torch.nn.Dropout(0.2), torch.nn.Linear(512, answer_classes))\\n        self.mask_head = torch.nn.Sequential(torch.nn.Conv2d(256, 128, 3, padding=1),\\n            torch.nn.GELU(), torch.nn.Conv2d(128, 1, 1))\\n\\n    def forward(self, time_a, time_b, tokens):\\n        feature_a, feature_b = self.stem(time_a), self.stem(time_b)\\n        fused = self.fuse(torch.cat([feature_a, feature_b, (feature_a-feature_b).abs(), feature_a*feature_b], 1))\\n        visual = F.adaptive_avg_pool2d(fused, 1).flatten(1)\\n        _, hidden = self.question(self.embedding(tokens))\\n        question = torch.cat([hidden[-2], hidden[-1]], 1)\\n        answer = self.answer_head(torch.cat([visual, question], 1))\\n        mask = F.interpolate(self.mask_head(fused), size=time_a.shape[-2:], mode=\\"bilinear\\", align_corners=False)\\n        return answer, mask\\n\\n\\nclass FusionExpert(torch.nn.Module):\\n    def __init__(self, backbone, embedding_dim: int, num_classes: int, image_size: int = 224):\\n        super().__init__()\\n        self.backbone = backbone\\n        self.norm = torch.nn.LayerNorm(embedding_dim)\\n        self.classifier = torch.nn.Linear(embedding_dim, num_classes)\\n        self.num_classes, self.image_size = num_classes, image_size\\n\\n    def forward(self, *, s2=None, s1=None):\\n        inputs = {}\\n        if s2 is not None:\\n            inputs[\\"S2L2A\\"] = s2\\n        if s1 is not None:\\n            inputs[\\"S1GRD\\"] = s1\\n        if not inputs:\\n            raise ValueError(\\"At least one modality is required\\")\\n        tokens = self.backbone(inputs)[-1]\\n        patch_logits = self.classifier(self.norm(tokens))\\n        class_logits = patch_logits.mean(1)\\n        side = math.isqrt(tokens.shape[1])\\n        if side * side != tokens.shape[1]:\\n            raise ValueError(\\"Backbone tokens cannot be reshaped to a square evidence grid\\")\\n        evidence = patch_logits.transpose(1, 2).reshape(tokens.shape[0], self.num_classes, side, side)\\n        evidence = F.interpolate(evidence, (self.image_size, self.image_size), mode=\\"bilinear\\", align_corners=False)\\n        return class_logits, evidence\\n", "satquery_ml/sensors.py": "\\"\\"\\"Product metadata recognition. No filename, resolution or band-count sensor guesses.\\n\\nThis dependency-free module is mirrored into the ML package and Kaggle patch by\\nscripts/sync-expert-notebooks.py. Embedded metadata is a declaration, not certification.\\n\\"\\"\\"\\n\\n\\nimport re\\n\\n\\ndef compact(value):\\n    return re.sub(r\\"[^a-z0-9]\\", \\"\\", str(value).lower())\\n\\n\\ndef sensor_profile(source):\\n    tags = dict(source.tags())\\n    for namespace in source.tag_namespaces()[:8]:\\n        if namespace not in {\\"IMAGE_STRUCTURE\\", \\"DERIVED_SUBDATASETS\\"}:\\n            tags.update(dict(list(source.tags(ns=namespace).items())[:60]))\\n    normalized = {compact(key): str(value).strip() for key, value in tags.items()}\\n    names = [\\n        normalized[key]\\n        for key in (\\"satid\\", \\"satellite\\", \\"platform\\", \\"satellitename\\")\\n        if key in normalized\\n    ]\\n    platforms = set()\\n    for name in names:\\n        value = compact(name)\\n        if value in {\\"eos04\\", \\"risat1a\\"}:\\n            platforms.add(\\"eos-04\\")\\n        elif value == \\"risat1\\":\\n            platforms.add(\\"risat-1\\")\\n        elif value in {\\"cartosat2s\\", \\"cartosat2e\\", \\"cartosat2f\\", \\"c2s\\", \\"c2e\\", \\"c2f\\"}:\\n            platforms.add(\\"cartosat-2-series\\")\\n        elif value in {\\"sentinel2\\", \\"sentinel2a\\", \\"sentinel2b\\", \\"sentinel2c\\", \\"s2a\\", \\"s2b\\", \\"s2c\\"}:\\n            platforms.add(\\"sentinel-2\\")\\n        elif value in {\\"sentinel1\\", \\"sentinel1a\\", \\"sentinel1b\\", \\"sentinel1c\\", \\"s1a\\", \\"s1b\\", \\"s1c\\"}:\\n            platforms.add(\\"sentinel-1\\")\\n    if len(platforms) > 1:\\n        raise ValueError(\\"Conflicting embedded platform declarations\\")\\n    platform = next(iter(platforms), \\"unknown\\")\\n    numeric = (\\n        {\\"b1\\": \\"blue\\", \\"b2\\": \\"green\\", \\"b3\\": \\"red\\", \\"b4\\": \\"nir\\"}\\n        if platform == \\"cartosat-2-series\\"\\n        else {\\n            \\"b2\\": \\"blue\\",\\n            \\"b3\\": \\"green\\",\\n            \\"b4\\": \\"red\\",\\n            \\"b8\\": \\"nir\\",\\n            \\"b11\\": \\"swir1\\",\\n            \\"b12\\": \\"swir2\\",\\n        }\\n        if platform == \\"sentinel-2\\"\\n        else {}\\n    )\\n    semantic = {\\n        \\"red\\": \\"red\\",\\n        \\"green\\": \\"green\\",\\n        \\"blue\\": \\"blue\\",\\n        \\"nir\\": \\"nir\\",\\n        \\"nearinfrared\\": \\"nir\\",\\n        \\"swir1\\": \\"swir1\\",\\n        \\"shortwaveinfrared1\\": \\"swir1\\",\\n        \\"swir2\\": \\"swir2\\",\\n        \\"shortwaveinfrared2\\": \\"swir2\\",\\n        \\"pan\\": \\"pan\\",\\n        \\"panchromatic\\": \\"pan\\",\\n    }\\n    bands = []\\n    for index, description in enumerate(source.descriptions, 1):\\n        band_tags = {\\n            compact(key): str(value) for key, value in list(source.tags(index).items())[:40]\\n        }\\n        declarations = [\\n            description or \\"\\",\\n            band_tags.get(\\"bandname\\", \\"\\"),\\n            band_tags.get(\\"description\\", \\"\\"),\\n        ]\\n        meanings, pols = set(), set()\\n        for declaration in declarations:\\n            value = compact(declaration)\\n            numeric_name = re.sub(r\\"^b0+\\", \\"b\\", value)\\n            meaning = semantic.get(value) or numeric.get(numeric_name)\\n            if meaning:\\n                meanings.add(meaning)\\n            if value.upper() in {\\"HH\\", \\"HV\\", \\"VH\\", \\"VV\\", \\"RH\\", \\"RV\\", \\"LH\\", \\"LV\\"}:\\n                pols.add(value.upper())\\n        color = source.colorinterp[index - 1].name\\n        if color in {\\"red\\", \\"green\\", \\"blue\\"}:\\n            meanings.add(color)\\n        pol = normalized.get(f\\"txrxpol{index}\\", band_tags.get(\\"polarization\\", \\"\\")).upper()\\n        if pol in {\\"HH\\", \\"HV\\", \\"VH\\", \\"VV\\", \\"RH\\", \\"RV\\", \\"LH\\", \\"LV\\"}:\\n            pols.add(pol)\\n        if len(meanings) > 1 or len(pols) > 1:\\n            raise ValueError(f\\"Conflicting band declarations at index {index}\\")\\n        bands.append(\\n            {\\n                \\"index\\": index,\\n                \\"description\\": str(description or \\"\\")[:160],\\n                \\"meaning\\": next(iter(meanings), \\"unknown\\"),\\n                \\"polarization\\": next(iter(pols), None),\\n                \\"color\\": color,\\n                \\"unit\\": source.units[index - 1],\\n                \\"scale\\": source.scales[index - 1],\\n                \\"offset\\": source.offsets[index - 1],\\n            }\\n        )\\n    known = [band[\\"meaning\\"] for band in bands if band[\\"meaning\\"] != \\"unknown\\"]\\n    if len(known) != len(set(known)):\\n        raise ValueError(\\"Duplicate spectral band meanings\\")\\n    return {\\n        \\"platform\\": platform,\\n        \\"source\\": \\"embedded_product_metadata\\",\\n        \\"sensor\\": normalized.get(\\"sensor\\", \\"unknown\\"),\\n        \\"product_type\\": normalized.get(\\"producttype\\", \\"unknown\\"),\\n        \\"imaging_mode\\": normalized.get(\\"imagingmode\\", \\"unknown\\"),\\n        \\"representation\\": normalized.get(\\"representation\\", \\"unknown\\"),\\n        \\"rtc_applied\\": {\\"0\\": False, \\"1\\": True}.get(normalized.get(\\"rtcapplyflag\\")),\\n        \\"bands\\": bands,\\n        \\"warning\\": \\"Declared metadata only; raster spacing does not establish native resolution.\\",\\n    }\\n\\n\\ndef semantic_indexes(source, meanings):\\n    bands = sensor_profile(source)[\\"bands\\"]\\n    result = []\\n    for meaning in meanings:\\n        matches = [band[\\"index\\"] for band in bands if band[\\"meaning\\"] == meaning]\\n        if len(matches) != 1:\\n            return None\\n        result.append(matches[0])\\n    return result\\n\\n\\ndef visual_indexes(source):\\n    return semantic_indexes(source, [\\"red\\", \\"green\\", \\"blue\\"]) or (\\n        [1, 2, 3] if source.count >= 3 else [1, 1, 1]\\n    )\\n\\n\\ndef sentinel_fusion_indexes(optical, sar):\\n    \\"\\"\\"TerraMind\'s training channels are not interchangeable with RISAT/Cartosat.\\"\\"\\"\\n    s2, s1 = sensor_profile(optical), sensor_profile(sar)\\n    if s2[\\"platform\\"] != \\"sentinel-2\\" or s1[\\"platform\\"] != \\"sentinel-1\\":\\n        raise ValueError(\\n            \\"Fusion requires Sentinel-2 and Sentinel-1; ISRO transfer is unvalidated\\"\\n        )\\n    order = [\\"B01\\", \\"B02\\", \\"B03\\", \\"B04\\", \\"B05\\", \\"B06\\", \\"B07\\", \\"B08\\", \\"B8A\\", \\"B09\\", \\"B11\\", \\"B12\\"]\\n    descriptions = [str(item or \\"\\").upper().strip() for item in optical.descriptions]\\n    if any(descriptions.count(name) != 1 for name in order):\\n        raise ValueError(\\"Explicit ordered Sentinel-2 band names required\\")\\n    pols = [band[\\"polarization\\"] for band in s1[\\"bands\\"]]\\n    if any(pols.count(pol) != 1 for pol in [\\"VV\\", \\"VH\\"]):\\n        raise ValueError(\\"This expert requires VV/VH; RH/RV or HH/HV cannot substitute\\")\\n    if s1[\\"representation\\"].lower() not in {\\"sigma0_db\\", \\"sigma0db\\"}:\\n        raise ValueError(\\"Calibrated sigma0 in dB must be declared; raw amplitude is unsupported\\")\\n    if s2[\\"representation\\"].lower() != \\"surface_reflectance_10000\\":\\n        raise ValueError(\\"S2 L2A reflectance scaled by 10000 must be declared\\")\\n    return [descriptions.index(name) + 1 for name in order], [\\n        pols.index(pol) + 1 for pol in [\\"VV\\", \\"VH\\"]\\n    ]\\n", "satquery_model_service/contracts.py": "from __future__ import annotations\\n\\nfrom typing import Any, Literal\\n\\nfrom pydantic import BaseModel, ConfigDict, Field\\n\\n\\nclass StrictModel(BaseModel):\\n    model_config = ConfigDict(extra=\\"forbid\\")\\n\\n\\nclass Step(StrictModel):\\n    step_id: str\\n    task: Literal[\\n        \\"single_vqa\\", \\"caption\\", \\"grounding\\", \\"change_vqa\\", \\"optical_sar_fusion\\"\\n    ]\\n    asset_ids: list[str]\\n    permitted_params: dict[str, Any]\\n    policy_reason: str\\n\\n\\nclass Asset(StrictModel):\\n    id: str\\n    original_name: str\\n    content_type: str\\n    size_bytes: int\\n    sha256: str\\n    role: str\\n    modality: str\\n    source_dataset: str | None = None\\n    created_at: str\\n    metadata: dict[str, Any] | None = None\\n    validation_errors: list[str] = Field(default_factory=list)\\n\\n\\nclass GeospatialContext(StrictModel):\\n    latitude: float = Field(ge=-90, le=90)\\n    longitude: float = Field(ge=-180, le=180)\\n    altitude_m: float | None = Field(default=None, ge=-500, le=100_000)\\n    captured_at: str | None = None\\n    sensor: str | None = Field(default=None, max_length=120)\\n    source: Literal[\\"user\\", \\"gps\\", \\"exif\\", \\"raster\\"] = \\"user\\"\\n    metadata: dict[str, str | int | float | bool] = Field(default_factory=dict)\\n\\n\\nclass InferencePayload(StrictModel):\\n    step: Step\\n    query: str = Field(min_length=2, max_length=2_000)\\n    assets: list[Asset] = Field(min_length=1, max_length=2)\\n    context: GeospatialContext | None = None\\n\\n\\nclass Evidence(StrictModel):\\n    id: str\\n    type: Literal[\\"box\\", \\"polygon\\", \\"mask\\", \\"heatmap\\", \\"text_region\\"]\\n    label: str\\n    score: float = Field(ge=0, le=1)\\n    coordinate_space: Literal[\\"normalized\\", \\"pixel\\", \\"geographic\\"]\\n    geometry: dict[str, Any]\\n    asset_id: str\\n    artifact_url: str | None = None\\n\\n\\nclass SpecialistResponse(StrictModel):\\n    task: str\\n    text: str\\n    facts: list[dict[str, Any]] = Field(default_factory=list)\\n    evidence: list[Evidence] = Field(default_factory=list)\\n    raw_score: float = Field(ge=0, le=1)\\n    score_kind: Literal[\\"calibrated_probability\\", \\"evidence_quality\\", \\"uncalibrated\\"]\\n    model_version: str\\n    warnings: list[str] = Field(default_factory=list)\\n", "satquery_model_service/paired_adapters.py": "\\"\\"\\"Strict runtime for the v2 cloud-notebook artifacts. No random-weight fallback.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport base64\\nimport hashlib\\nimport io\\nimport json\\nimport re\\nfrom pathlib import Path\\n\\nimport numpy as np\\n\\nfrom satquery_model_service.contracts import Evidence, SpecialistResponse\\n\\n\\ndef sha256(path):\\n    digest = hashlib.sha256()\\n    with path.open(\\"rb\\") as handle:\\n        for block in iter(lambda: handle.read(1024 * 1024), b\\"\\"):\\n            digest.update(block)\\n    return digest.hexdigest()\\n\\n\\ndef verified_config(root):\\n    manifest_path = root / \\"sha256_manifest.json\\"\\n    manifest = json.loads(manifest_path.read_text(encoding=\\"utf-8\\"))\\n    for name in (\\"model.safetensors\\", \\"config.json\\"):\\n        if manifest.get(name) != sha256(root / name):\\n            raise ValueError(f\\"Artifact integrity check failed: {name}\\")\\n    config = json.loads((root / \\"config.json\\").read_text(encoding=\\"utf-8\\"))\\n    if config.get(\\"artifact_version\\") != \\"satquery-pair-v2\\":\\n        raise ValueError(\\n            \\"Use the corrected v2 training notebook export; legacy architectures are incompatible\\"\\n        )\\n    return config, f\\"paired-v2:sha256:{sha256(root / \'model.safetensors\')}\\"\\n\\n\\ndef validate_pair(paths):\\n    import rasterio\\n\\n    with rasterio.open(paths[0]) as a, rasterio.open(paths[1]) as b:\\n        if (a.width, a.height, a.crs) != (\\n            b.width,\\n            b.height,\\n            b.crs,\\n        ) or not a.transform.almost_equals(b.transform):\\n            raise ValueError(\\n                \\"Learned expert requires an identical co-registered source grid\\"\\n            )\\n        if a.width * a.height > 4_194_304:\\n            raise ValueError(\\n                \\"Tile this scene to at most 2048\\u00d72048 pixels before learned pair inference\\"\\n            )\\n        for source in (a, b):\\n            if any(\\"complex\\" in value for value in source.dtypes):\\n                raise ValueError(\\n                    \\"Complex SAR cannot be converted to real intensity implicitly\\"\\n                )\\n        return a.height, a.width\\n\\n\\ndef require_shared_support(valid):\\n    if not valid.any():\\n        raise ValueError(\\"Paired rasters contain no shared valid pixels\\")\\n\\n\\ndef resample_supported_mask(candidate, valid):\\n    from PIL import Image\\n\\n    height, width = valid.shape\\n    scale = min(1, 1024 / max(height, width))\\n    shape = (max(1, round(width * scale)), max(1, round(height * scale)))\\n    candidate = np.asarray(\\n        Image.fromarray(candidate).resize(shape, Image.Resampling.NEAREST)\\n    )\\n    support = np.asarray(Image.fromarray(valid).resize(shape, Image.Resampling.NEAREST))\\n    # PIL-backed arrays are read-only; an in-place &= would fail on every request.\\n    return candidate & support\\n\\n\\nclass ChangeAdapter:\\n    capability = \\"change\\"\\n\\n    def __init__(self, settings):\\n        import torch\\n        from safetensors.torch import load_file\\n        from satquery_ml.models.notebook_experts import ChangeExpert\\n\\n        root = Path(settings.artifact_dir or \\"\\")\\n        config, self.version = verified_config(root)\\n        if config.get(\\"architecture\\") != \\"shared_resnet18_gru_answer_mask\\":\\n            raise ValueError(\\"Wrong change architecture\\")\\n        self.vocabulary = config[\\"word_vocab\\"]\\n        self.answers = [\\n            name\\n            for name, _ in sorted(\\n                config[\\"answer_vocab\\"].items(), key=lambda row: row[1]\\n            )\\n        ]\\n        self.size = int(config[\\"config\\"][\\"image_size\\"])\\n        self.max_tokens = int(config[\\"config\\"][\\"max_question_tokens\\"])\\n        self.device = torch.device(\\n            settings.device if torch.cuda.is_available() else \\"cpu\\"\\n        )\\n        self.model = ChangeExpert(\\n            len(self.vocabulary), len(self.answers), pretrained=False\\n        )\\n        self.model.load_state_dict(load_file(root / \\"model.safetensors\\"), strict=True)\\n        self.model.to(self.device).eval()\\n\\n    def infer(self, payload, paths):\\n        import rasterio\\n        import torch\\n        from PIL import Image\\n        from satquery_ml.sensors import visual_indexes\\n\\n        if payload.step.task != \\"change_vqa\\" or len(paths) != 2:\\n            raise ValueError(\\"ChangeVQA requires two temporal scenes\\")\\n        pairs = {\\n            asset.role: (asset, path)\\n            for asset, path in zip(payload.assets, paths, strict=True)\\n        }\\n        if set(pairs) != {\\"time_a\\", \\"time_b\\"}:\\n            raise ValueError(\\"Explicit time_a and time_b roles required\\")\\n        ordered = [pairs[role] for role in (\\"time_a\\", \\"time_b\\")]\\n        height, width = validate_pair([pair[1] for pair in ordered])\\n        tensors, valid = [], np.ones((height, width), dtype=bool)\\n        for asset, path in ordered:\\n            if asset.modality not in {\\"optical\\", \\"multispectral\\"}:\\n                raise ValueError(\\n                    \\"The CDVQA/SECOND expert is trained on optical RGB, not SAR\\"\\n                )\\n            with rasterio.open(path) as source:\\n                indexes = visual_indexes(source)\\n                if source.count < 3 or any(\\n                    source.dtypes[index - 1] != \\"uint8\\" for index in indexes\\n                ):\\n                    raise ValueError(\\n                        \\"CDVQA expert expects uint8 RGB previews, not unnormalized spectral DN\\"\\n                    )\\n                raw = source.read(indexes, masked=True)\\n                valid &= ~np.ma.getmaskarray(raw).any(axis=0) & (\\n                    source.dataset_mask() > 0\\n                )\\n                image = Image.fromarray(np.moveaxis(raw.filled(0), 0, -1)).resize(\\n                    (self.size, self.size), Image.Resampling.BILINEAR\\n                )\\n                array = np.asarray(image).astype(np.float32).transpose(2, 0, 1) / 255.0\\n                array = (\\n                    array\\n                    - np.array([0.485, 0.456, 0.406], dtype=np.float32)[:, None, None]\\n                ) / np.array([0.229, 0.224, 0.225], dtype=np.float32)[:, None, None]\\n                tensors.append(torch.from_numpy(array).unsqueeze(0).to(self.device))\\n        require_shared_support(valid)\\n        tokens = [\\n            self.vocabulary.get(token, 1)\\n            for token in re.findall(r\\"[a-z0-9\']+\\", payload.query.lower())\\n        ]\\n        tokens = (tokens[: self.max_tokens] + [0] * self.max_tokens)[: self.max_tokens]\\n        with torch.inference_mode():\\n            logits, mask_logits = self.model(\\n                *tensors, torch.tensor([tokens], device=self.device)\\n            )\\n            if (\\n                not torch.isfinite(logits).all()\\n                or not torch.isfinite(mask_logits).all()\\n            ):\\n                raise ValueError(\\"Non-finite learned output\\")\\n            probabilities = logits.softmax(1)[0]\\n            index, score = int(probabilities.argmax()), float(probabilities.max())\\n            candidate = mask_logits.sigmoid()[0, 0].cpu().numpy() >= 0.5\\n        candidate = resample_supported_mask(candidate, valid)\\n        shape = candidate.shape[::-1]\\n        stream = io.BytesIO()\\n        Image.fromarray(candidate.astype(np.uint8) * 255).save(stream, format=\\"PNG\\")\\n        return SpecialistResponse(\\n            task=\\"change_vqa\\",\\n            text=f\\"Learned closed-vocabulary ChangeVQA answer: {self.answers[index]}. \\"\\n            \\"The separate mask predicts generic semantic change, not a target-specific loss or event cause.\\",\\n            facts=[\\n                {\\"name\\": \\"answer_label\\", \\"value\\": self.answers[index]},\\n                {\\"name\\": \\"execution_mode\\", \\"value\\": \\"learned_paired_change\\"},\\n            ],\\n            evidence=[\\n                Evidence(\\n                    id=\\"ev_learned_change\\",\\n                    type=\\"mask\\",\\n                    label=\\"Learned semantic-change candidate\\",\\n                    score=score,\\n                    coordinate_space=\\"pixel\\",\\n                    asset_id=ordered[1][0].id,\\n                    geometry={\\n                        \\"encoding\\": \\"png-base64\\",\\n                        \\"data\\": base64.b64encode(stream.getvalue()).decode(),\\n                        \\"width\\": shape[0],\\n                        \\"height\\": shape[1],\\n                        \\"method\\": \\"CDVQA SECOND supervised change mask\\",\\n                        \\"target\\": \\"semantic_change\\",\\n                        \\"threshold\\": 0.5,\\n                        \\"status\\": \\"candidate\\",\\n                        \\"comparison_asset_id\\": ordered[0][0].id,\\n                    },\\n                )\\n            ],\\n            raw_score=score,\\n            score_kind=\\"uncalibrated\\",\\n            model_version=self.version,\\n            warnings=[\\n                \\"Answer softmax is not calibrated. SECOND-domain validation does not establish Cartosat or Nepal-flood accuracy.\\",\\n                \\"Mask boundaries are upsampled model estimates, not source-resolution delineation.\\",\\n            ],\\n        )\\n\\n\\nclass FusionAdapter:\\n    capability = \\"fusion\\"\\n\\n    def __init__(self, settings):\\n        import torch\\n        from safetensors.torch import load_file\\n        from terratorch.registry import BACKBONE_REGISTRY\\n        from satquery_ml.models.notebook_experts import FusionExpert\\n\\n        root = Path(settings.artifact_dir or \\"\\")\\n        config, self.version = verified_config(root)\\n        if config.get(\\"mask_supervision\\") != \\"none; scene labels only\\":\\n            raise ValueError(\\"Unknown fusion artifact supervision contract\\")\\n        self.labels, self.normalization = config[\\"classes\\"], config[\\"normalization\\"]\\n        self.size = int(config[\\"config\\"][\\"image_size\\"])\\n        self.device = torch.device(\\n            settings.device if torch.cuda.is_available() else \\"cpu\\"\\n        )\\n        backbone = BACKBONE_REGISTRY.build(\\n            config[\\"backbone\\"],\\n            pretrained=False,\\n            modalities=[\\"S2L2A\\", \\"S1GRD\\"],\\n            merge_method=\\"mean\\",\\n        )\\n        self.model = FusionExpert(\\n            backbone, int(config[\\"embedding_dim\\"]), len(self.labels), self.size\\n        )\\n        self.model.load_state_dict(load_file(root / \\"model.safetensors\\"), strict=True)\\n        self.model.to(self.device).eval()\\n\\n    def infer(self, payload, paths):\\n        import rasterio\\n        import torch\\n        from torch.nn import functional as F\\n        from satquery_ml.sensors import sentinel_fusion_indexes\\n\\n        if payload.step.task != \\"optical_sar_fusion\\" or len(paths) != 2:\\n            raise ValueError(\\"Fusion requires exactly two registered modalities\\")\\n        pairs = [\\n            (asset, path) for asset, path in zip(payload.assets, paths, strict=True)\\n        ]\\n        optical = next(\\n            (\\n                path\\n                for asset, path in pairs\\n                if asset.modality in {\\"optical\\", \\"multispectral\\"}\\n            ),\\n            None,\\n        )\\n        sar = next((path for asset, path in pairs if asset.modality == \\"sar\\"), None)\\n        if optical is None or sar is None:\\n            raise ValueError(\\"Declare optical and SAR modalities\\")\\n        validate_pair([optical, sar])\\n        with rasterio.open(optical) as s2, rasterio.open(sar) as s1:\\n            indexes = sentinel_fusion_indexes(s2, s1)\\n            inputs = []\\n            for source, bands, prefix in zip(\\n                [s2, s1], indexes, [\\"s2\\", \\"s1\\"], strict=True\\n            ):\\n                raw = source.read(bands, masked=True).astype(np.float32)\\n                if (\\n                    np.ma.getmaskarray(raw).any()\\n                    or not np.isfinite(raw.data).all()\\n                    or not (source.dataset_mask() > 0).all()\\n                ):\\n                    raise ValueError(\\n                        \\"Use a shared-valid cropped fusion tile; missing pixels cannot be fabricated\\"\\n                    )\\n                array = torch.from_numpy(raw.data)\\n                array = F.interpolate(\\n                    array[None],\\n                    (self.size, self.size),\\n                    mode=\\"bilinear\\",\\n                    align_corners=False,\\n                )[0]\\n                mean = torch.tensor(self.normalization[f\\"{prefix}_mean\\"])[:, None, None]\\n                std = torch.tensor(self.normalization[f\\"{prefix}_std\\"])[:, None, None]\\n                inputs.append(((array - mean) / std)[None].to(self.device))\\n        with torch.inference_mode():\\n            logits, _ = self.model(s2=inputs[0], s1=inputs[1])\\n            if not torch.isfinite(logits).all():\\n                raise ValueError(\\"Non-finite fusion predictions\\")\\n            scores = logits.sigmoid()[0].cpu().numpy()\\n        selected = np.flatnonzero(scores >= 0.5)\\n        labels = \\", \\".join(self.labels[index] for index in selected)\\n        return SpecialistResponse(\\n            task=\\"optical_sar_fusion\\",\\n            text=f\\"Learned TerraMind optical/SAR scene-label candidates: {labels}.\\"\\n            if labels\\n            else \\"No learned scene label exceeded the default 0.5 threshold.\\",\\n            facts=[\\n                {\\n                    \\"name\\": \\"scene_class_scores\\",\\n                    \\"value\\": dict(zip(self.labels, scores.tolist(), strict=True)),\\n                },\\n                {\\"name\\": \\"execution_mode\\", \\"value\\": \\"learned_cross_modal_fusion\\"},\\n            ],\\n            evidence=[],\\n            raw_score=float(scores.max()),\\n            score_kind=\\"uncalibrated\\",\\n            model_version=self.version,\\n            warnings=[\\n                \\"This artifact has scene-label supervision only: no precision masks are returned.\\",\\n                \\"Sigmoid scores are uncalibrated; Cartosat and RISAT transfer requires separate training/evaluation.\\",\\n            ],\\n        )\\n"}')
for name, source in runtime_files.items():
    target = PAIR_RUNTIME_ROOT / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(source, encoding='utf-8')
if str(PAIR_RUNTIME_ROOT) not in sys.path:
    sys.path.insert(0, str(PAIR_RUNTIME_ROOT))

In [ ]:
# OPTIONAL: only after training/exporting v2 ChangeVQA or TerraMind artifacts.
# Attach each exported artifact folder as a private Kaggle input or copy it into /kaggle/working.
# Never populate these paths with random/untrained weights. Empty values leave Qwen unchanged.
PAIR_ARTIFACT_DIRS = {"change": "", "fusion": ""}

from pathlib import Path
from types import SimpleNamespace
import tempfile

from satquery_model_service.paired_adapters import ChangeAdapter, FusionAdapter
from satquery_model_service.contracts import InferencePayload as PairInferencePayload

assert not inference_lock.locked(), "Wait for the active request to finish."
pair_adapters = {}
for capability, raw_path in PAIR_ARTIFACT_DIRS.items():
    if not raw_path:
        continue
    root = Path(raw_path).resolve()
    if not any(root.is_relative_to(Path(prefix)) for prefix in ("/kaggle/input", "/kaggle/working")):
        raise ValueError("Pair artifact must be inside an explicitly attached Kaggle input/working folder")
    cls = ChangeAdapter if capability == "change" else FusionAdapter
    pair_adapters[capability] = cls(SimpleNamespace(artifact_dir=root, device="cuda:0"))
    print(f"{capability}: strict artifact load passed. Run a real paired HTTP smoke test before enabling it in the backend.")


async def paired_infer(task: str, payload: Annotated[str, Form()],
                       assets: Annotated[list[UploadFile], File()],
                       authorization: Annotated[str | None, Header()] = None):
    if task not in {"change_vqa", "optical_sar_fusion"}:
        return await quality_infer(task, payload, assets, authorization)
    authorize(authorization)
    capability = "change" if task == "change_vqa" else "fusion"
    if capability not in pair_adapters:
        raise HTTPException(409, f"No trained {capability} artifact loaded. No analytical fallback is disguised as learned inference.")
    try:
        contract = PairInferencePayload.model_validate_json(payload)
        if task != contract.step.task or len(assets) != 2 or len(contract.assets) != 2:
            raise ValueError("A pair task requires exactly two matching assets")
        if contract.step.asset_ids != [item.id for item in contract.assets]:
            raise ValueError("Asset order/IDs must match the declared step")
        async with inference_lock:
            with tempfile.TemporaryDirectory(prefix="satquery-pair-") as directory:
                paths = []
                for index, upload in enumerate(assets):
                    data = await upload.read(MAX_UPLOAD_BYTES + 1)
                    await upload.close()
                    if not data or len(data) > MAX_UPLOAD_BYTES:
                        raise ValueError("Invalid pair upload size")
                    if hashlib.sha256(data).hexdigest() != contract.assets[index].sha256:
                        raise ValueError("Uploaded bytes do not match the declared asset hash")
                    path = Path(directory) / f"asset-{index}.tif"
                    path.write_bytes(data)
                    paths.append(path)
                result = await asyncio.to_thread(pair_adapters[capability].infer, contract, paths)
                return result.model_dump()
    except ValueError as exc:
        raise HTTPException(422, str(exc)) from exc


async def combined_ready():
    return {"status": "ready", "capability": "vlm", "capabilities": ["vlm", *pair_adapters],
            "model_version": MODEL_VERSION, "quality_pipeline": QUALITY_VERSION,
            "planner": "qwen-intent-v1" if "learned_plan" in globals() else None,
            "pair_artifacts": {key: value.version for key, value in pair_adapters.items()}}


for route in app.routes:
    if getattr(route, "path", None) == "/v1/infer/{task}":
        route.endpoint = route.dependant.call = paired_infer
    elif getattr(route, "path", None) == "/ready":
        route.endpoint = route.dependant.call = combined_ready
app.openapi_schema = None
print("Optional paired routes installed. Trained capabilities:", sorted(pair_adapters))
print("Qwen remains available. No paid endpoint, new tunnel, training job or automatic fallback was created.")